<a href="https://colab.research.google.com/github/mujasss/Tugas-4-Sistem-temu-kembali/blob/main/240210502003_Muja_Adila_Setiawan_TFIDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Tugas 4 nomor 2

In [14]:
# Install library yang dibutuhkan: Sastrawi (stopwords), scikit-learn, pandas
!pip install Sastrawi scikit-learn pandas -q

In [15]:
import re
import math
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Inisialisasi stopword remover Sastrawi
_stopword_remover = StopWordRemoverFactory().create_stop_word_remover()


def simple_preprocess(text):
    """
    Poin 1: Preprocessing sederhana.
    Minimal terdiri dari: case folding + tokenisasi + stopwords removal.
    """
    text = text.lower()                          # case folding
    text = re.sub(r"[^a-z\s]", " ", text)         # cleaning ringan (hapus tanda baca/angka)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()                         # tokenisasi
    tokens = _stopword_remover.remove(" ".join(tokens)).split()  # stopwords removal
    return tokens

In [16]:
# -----------------------------------------------------------------------------
# Dataset: minimal 4 dokumen pendek berbahasa Indonesia (1-3 kalimat),
# tema sistem komputer, jaringan, dan kecerdasan buatan sesuai studi kasus soal.
# -----------------------------------------------------------------------------
dokumen_asli = [
    "Sistem temu kembali informasi membantu pengguna menemukan dokumen yang relevan.",
    "Jaringan komputer menghubungkan banyak perangkat agar dapat saling bertukar data.",
    "Kecerdasan buatan mempelajari pola data untuk membuat keputusan secara otomatis.",
    "Sistem komputer modern menggunakan kecerdasan buatan untuk mempercepat pencarian informasi.",
]

# Terapkan preprocessing sederhana ke setiap dokumen
dokumen_tokens = [simple_preprocess(d) for d in dokumen_asli]

print("=" * 80)
print("POIN 1: HASIL PREPROCESSING SEDERHANA")
print("=" * 80)
for i, (asli, tok) in enumerate(zip(dokumen_asli, dokumen_tokens), start=1):
    print("Dok" + str(i), "asli :", asli)
    print("Dok" + str(i), "token:", tok)
    print()

POIN 1: HASIL PREPROCESSING SEDERHANA
Dok1 asli : Sistem temu kembali informasi membantu pengguna menemukan dokumen yang relevan.
Dok1 token: ['sistem', 'temu', 'informasi', 'membantu', 'pengguna', 'menemukan', 'dokumen', 'relevan']

Dok2 asli : Jaringan komputer menghubungkan banyak perangkat agar dapat saling bertukar data.
Dok2 token: ['jaringan', 'komputer', 'menghubungkan', 'banyak', 'perangkat', 'dapat', 'saling', 'bertukar', 'data']

Dok3 asli : Kecerdasan buatan mempelajari pola data untuk membuat keputusan secara otomatis.
Dok3 token: ['kecerdasan', 'buatan', 'mempelajari', 'pola', 'data', 'membuat', 'keputusan', 'otomatis']

Dok4 asli : Sistem komputer modern menggunakan kecerdasan buatan untuk mempercepat pencarian informasi.
Dok4 token: ['sistem', 'komputer', 'modern', 'menggunakan', 'kecerdasan', 'buatan', 'mempercepat', 'pencarian', 'informasi']



In [17]:
# -----------------------------------------------------------------------------
# Poin 2: Bangun representasi Bag-of-Words (raw count) dari ke-4 dokumen.
# Vocabulary = seluruh term unik dari keempat dokumen.
# -----------------------------------------------------------------------------
vocabulary = sorted(set(token for tokens in dokumen_tokens for token in tokens))

bow = []
for tokens in dokumen_tokens:
    # hitung berapa kali tiap term dalam vocabulary muncul di dokumen ini
    row = {term: tokens.count(term) for term in vocabulary}
    bow.append(row)

df_bow = pd.DataFrame(bow, index=["Dok" + str(i + 1) for i in range(len(dokumen_tokens))])

print("=" * 80)
print("POIN 2: BAG-OF-WORDS (RAW COUNT)")
print("=" * 80)
print(df_bow.to_string())

POIN 2: BAG-OF-WORDS (RAW COUNT)
      banyak  bertukar  buatan  dapat  data  dokumen  informasi  jaringan  kecerdasan  keputusan  komputer  membantu  membuat  mempelajari  mempercepat  menemukan  menggunakan  menghubungkan  modern  otomatis  pencarian  pengguna  perangkat  pola  relevan  saling  sistem  temu
Dok1       0         0       0      0     0        1          1         0           0          0         0         1        0            0            0          1            0              0       0         0          0         1          0     0        1       0       1     1
Dok2       1         1       0      1     1        0          0         1           0          0         1         0        0            0            0          0            0              1       0         0          0         0          1     0        0       1       0     0
Dok3       0         0       1      0     1        0          0         0           1          1         0         0        1        

In [18]:
# -----------------------------------------------------------------------------
# Poin 3a: Hitung Term Frequency (TF) untuk setiap term di setiap dokumen.
# TF(t,d) = jumlah kemunculan term t di dokumen d / total token di dokumen d
# -----------------------------------------------------------------------------
N = len(dokumen_tokens)  # jumlah dokumen dalam koleksi (dipakai lagi di sel-sel berikutnya)

df_tf = df_bow.div(df_bow.sum(axis=1), axis=0)

print("=" * 80)
print("POIN 3a: TERM FREQUENCY (TF) MANUAL -> TF(t,d) = count(t,d) / total_token(d)")
print("=" * 80)
print(df_tf.round(4).to_string())

POIN 3a: TERM FREQUENCY (TF) MANUAL -> TF(t,d) = count(t,d) / total_token(d)
      banyak  bertukar  buatan   dapat    data  dokumen  informasi  jaringan  kecerdasan  keputusan  komputer  membantu  membuat  mempelajari  mempercepat  menemukan  menggunakan  menghubungkan  modern  otomatis  pencarian  pengguna  perangkat   pola  relevan  saling  sistem   temu
Dok1  0.0000    0.0000  0.0000  0.0000  0.0000    0.125     0.1250    0.0000      0.0000      0.000    0.0000     0.125    0.000        0.000       0.0000      0.125       0.0000         0.0000  0.0000     0.000     0.0000     0.125     0.0000  0.000    0.125  0.0000  0.1250  0.125
Dok2  0.1111    0.1111  0.0000  0.1111  0.1111    0.000     0.0000    0.1111      0.0000      0.000    0.1111     0.000    0.000        0.000       0.0000      0.000       0.0000         0.1111  0.0000     0.000     0.0000     0.000     0.1111  0.000    0.000  0.1111  0.0000  0.000
Dok3  0.0000    0.0000  0.1250  0.0000  0.1250    0.000     0.0000    0.00

In [19]:
# -----------------------------------------------------------------------------
# Poin 3b: Hitung Document Frequency (df) dan Inverse Document Frequency (IDF).
# df(t)  = jumlah dokumen yang mengandung term t
# IDF(t) = log( (1+N) / (1+df(t)) ) + 1  -> formula dengan smoothing,
#          sama seperti default TfidfVectorizer scikit-learn agar hasil
#          manual & library bisa dibandingkan secara adil di poin 4.
# -----------------------------------------------------------------------------
df_doc_freq = (df_bow > 0).sum(axis=0)                          # df(t)
idf_manual = ((1 + N) / (1 + df_doc_freq)).apply(math.log) + 1  # IDF(t)

df_idf = pd.DataFrame({"df(t)": df_doc_freq, "IDF(t)": idf_manual.round(4)})

print("=" * 80)
print("POIN 3b: DOCUMENT FREQUENCY (df) & INVERSE DOCUMENT FREQUENCY (IDF) MANUAL")
print("         IDF(t) = log( (1+N) / (1+df(t)) ) + 1   [N =", N, "dokumen]")
print("=" * 80)
print(df_idf.to_string())

POIN 3b: DOCUMENT FREQUENCY (df) & INVERSE DOCUMENT FREQUENCY (IDF) MANUAL
         IDF(t) = log( (1+N) / (1+df(t)) ) + 1   [N = 4 dokumen]
               df(t)  IDF(t)
banyak             1  1.9163
bertukar           1  1.9163
buatan             2  1.5108
dapat              1  1.9163
data               2  1.5108
dokumen            1  1.9163
informasi          2  1.5108
jaringan           1  1.9163
kecerdasan         2  1.5108
keputusan          1  1.9163
komputer           2  1.5108
membantu           1  1.9163
membuat            1  1.9163
mempelajari        1  1.9163
mempercepat        1  1.9163
menemukan          1  1.9163
menggunakan        1  1.9163
menghubungkan      1  1.9163
modern             1  1.9163
otomatis           1  1.9163
pencarian          1  1.9163
pengguna           1  1.9163
perangkat          1  1.9163
pola               1  1.9163
relevan            1  1.9163
saling             1  1.9163
sistem             2  1.5108
temu               1  1.9163


In [20]:
# -----------------------------------------------------------------------------
# Poin 3c: Hitung nilai TF-IDF untuk setiap term di setiap dokumen.
# TF-IDF(t,d) = TF(t,d) x IDF(t)
# Poin 5 (bagian manual): tampilkan matriks TF-IDF hasil perhitungan manual.
# -----------------------------------------------------------------------------
df_tfidf_manual = df_tf.mul(idf_manual, axis=1)

print("=" * 80)
print("POIN 3c & 5: MATRIKS TF-IDF MANUAL (TF x IDF)")
print("=" * 80)
print(df_tfidf_manual.round(4).to_string())

POIN 3c & 5: MATRIKS TF-IDF MANUAL (TF x IDF)
      banyak  bertukar  buatan   dapat    data  dokumen  informasi  jaringan  kecerdasan  keputusan  komputer  membantu  membuat  mempelajari  mempercepat  menemukan  menggunakan  menghubungkan  modern  otomatis  pencarian  pengguna  perangkat    pola  relevan  saling  sistem    temu
Dok1  0.0000    0.0000  0.0000  0.0000  0.0000   0.2395     0.1889    0.0000      0.0000     0.0000    0.0000    0.2395   0.0000       0.0000       0.0000     0.2395       0.0000         0.0000  0.0000    0.0000     0.0000    0.2395     0.0000  0.0000   0.2395  0.0000  0.1889  0.2395
Dok2  0.2129    0.2129  0.0000  0.2129  0.1679   0.0000     0.0000    0.2129      0.0000     0.0000    0.1679    0.0000   0.0000       0.0000       0.0000     0.0000       0.0000         0.2129  0.0000    0.0000     0.0000    0.0000     0.2129  0.0000   0.0000  0.2129  0.0000  0.0000
Dok3  0.0000    0.0000  0.1889  0.0000  0.1889   0.0000     0.0000    0.0000      0.1889     0.2395

In [21]:
# -----------------------------------------------------------------------------
# Poin 4: Implementasikan perhitungan TF-IDF menggunakan scikit-learn
# (TfidfVectorizer), sebagai pembanding terhadap hasil manual di atas.
# Poin 5 (bagian library): tampilkan matriks TF-IDF hasil scikit-learn.
# -----------------------------------------------------------------------------
# Gabungkan kembali token hasil preprocessing menjadi string agar vocabulary
# yang dipakai TfidfVectorizer konsisten dengan hasil preprocessing manual.
dokumen_bersih = [" ".join(tokens) for tokens in dokumen_tokens]

# norm=None dipakai agar mudah dibandingkan langsung dengan TF-IDF manual (tanpa normalisasi L2)
vectorizer = TfidfVectorizer(norm=None)
tfidf_matrix = vectorizer.fit_transform(dokumen_bersih)

df_tfidf_sklearn = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=["Dok" + str(i + 1) for i in range(len(dokumen_tokens))],
)

print("=" * 80)
print("POIN 4 & 5: MATRIKS TF-IDF DARI SCIKIT-LEARN (TfidfVectorizer, norm=None)")
print("=" * 80)
print(df_tfidf_sklearn.round(4).to_string())

# Versi tambahan dengan normalisasi L2 (default scikit-learn), berguna untuk
# perhitungan cosine similarity pada tahap sistem pencarian selanjutnya.
vectorizer_l2 = TfidfVectorizer()
tfidf_matrix_l2 = vectorizer_l2.fit_transform(dokumen_bersih)
df_tfidf_sklearn_l2 = pd.DataFrame(
    tfidf_matrix_l2.toarray(),
    columns=vectorizer_l2.get_feature_names_out(),
    index=["Dok" + str(i + 1) for i in range(len(dokumen_tokens))],
)

print("\n(Versi dengan normalisasi L2 / default scikit-learn, untuk cosine similarity):")
print(df_tfidf_sklearn_l2.round(4).to_string())

POIN 4 & 5: MATRIKS TF-IDF DARI SCIKIT-LEARN (TfidfVectorizer, norm=None)
      banyak  bertukar  buatan   dapat    data  dokumen  informasi  jaringan  kecerdasan  keputusan  komputer  membantu  membuat  mempelajari  mempercepat  menemukan  menggunakan  menghubungkan  modern  otomatis  pencarian  pengguna  perangkat    pola  relevan  saling  sistem    temu
Dok1  0.0000    0.0000  0.0000  0.0000  0.0000   1.9163     1.5108    0.0000      0.0000     0.0000    0.0000    1.9163   0.0000       0.0000       0.0000     1.9163       0.0000         0.0000  0.0000    0.0000     0.0000    1.9163     0.0000  0.0000   1.9163  0.0000  1.5108  1.9163
Dok2  1.9163    1.9163  0.0000  1.9163  1.5108   0.0000     0.0000    1.9163      0.0000     0.0000    1.5108    0.0000   0.0000       0.0000       0.0000     0.0000       0.0000         1.9163  0.0000    0.0000     0.0000    0.0000     1.9163  0.0000   0.0000  1.9163  0.0000  0.0000
Dok3  0.0000    0.0000  1.5108  0.0000  1.5108   0.0000     0.0000    0

In [22]:
# -----------------------------------------------------------------------------
# Poin 4 (lanjutan): Bandingkan hasil perhitungan manual dengan scikit-learn.
# Selisih dihitung secara absolut antar kedua matriks TF-IDF (norm=None).
# -----------------------------------------------------------------------------
selisih = (df_tfidf_manual[sorted(df_tfidf_manual.columns)] -
           df_tfidf_sklearn[sorted(df_tfidf_sklearn.columns)]).abs()

print("=" * 80)
print("POIN 4: SELISIH ABSOLUT ANTARA TF-IDF MANUAL vs SCIKIT-LEARN (norm=None)")
print("=" * 80)
print(selisih.round(4).to_string())
print("\nCatatan: selisih muncul karena definisi TF manual di sini dinormalisasi")
print("terhadap panjang dokumen, sedangkan TF bawaan scikit-learn memakai raw count.")
print("Nilai IDF keduanya sudah identik (sama-sama pakai formula smoothing).")

POIN 4: SELISIH ABSOLUT ANTARA TF-IDF MANUAL vs SCIKIT-LEARN (norm=None)
      banyak  bertukar  buatan   dapat   data  dokumen  informasi  jaringan  kecerdasan  keputusan  komputer  membantu  membuat  mempelajari  mempercepat  menemukan  menggunakan  menghubungkan  modern  otomatis  pencarian  pengguna  perangkat    pola  relevan  saling  sistem    temu
Dok1  0.0000    0.0000   0.000  0.0000  0.000   1.6768      1.322    0.0000       0.000     0.0000     0.000    1.6768   0.0000       0.0000       0.0000     1.6768       0.0000         0.0000  0.0000    0.0000     0.0000    1.6768     0.0000  0.0000   1.6768  0.0000   1.322  1.6768
Dok2  1.7034    1.7034   0.000  1.7034  1.343   0.0000      0.000    1.7034       0.000     0.0000     1.343    0.0000   0.0000       0.0000       0.0000     0.0000       0.0000         1.7034  0.0000    0.0000     0.0000    0.0000     1.7034  0.0000   0.0000  1.7034   0.000  0.0000
Dok3  0.0000    0.0000   1.322  0.0000  1.322   0.0000      0.000    0.0000

In [12]:
# -----------------------------------------------------------------------------
# Poin 6: Term manakah yang memiliki bobot TF-IDF tertinggi pada masing-masing
# dokumen? Mengapa term tersebut penting? (dijawab dalam analisis di bawah)
# -----------------------------------------------------------------------------
print("=" * 80)
print("POIN 6: TERM DENGAN BOBOT TF-IDF TERTINGGI PER DOKUMEN (scikit-learn, norm=None)")
print("=" * 80)
for idx, row in df_tfidf_sklearn.iterrows():
    top_term = row.idxmax()
    top_value = row.max()
    print(idx + ": '" + top_term + "' (bobot = {:.4f})".format(top_value))

analisis = """
ANALISIS SINGKAT:
Term dengan bobot TF-IDF tertinggi pada masing-masing dokumen cenderung merupakan kata
yang cukup sering muncul di dokumen tersebut namun jarang muncul di dokumen lain dalam
koleksi (df rendah), misalnya kata-kata topikal seperti jaringan, kecerdasan, atau
pola yang hanya muncul pada satu-dua dokumen. Sebaliknya, kata seperti sistem atau
komputer yang muncul di beberapa dokumen mendapat IDF lebih rendah sehingga bobot
TF-IDF-nya ditekan, meskipun frekuensi kemunculannya (TF) cukup tinggi. Hal ini
menunjukkan term-term tersebut penting karena bersifat diskriminatif: keberadaannya
membantu sistem membedakan topik satu dokumen dari dokumen lainnya, yang merupakan
fungsi utama dari pembobotan TF-IDF dalam sistem temu kembali informasi.
"""
print(analisis)

POIN 6: TERM DENGAN BOBOT TF-IDF TERTINGGI PER DOKUMEN (scikit-learn, norm=None)
Dok1: 'dokumen' (bobot = 1.9163)
Dok2: 'banyak' (bobot = 1.9163)
Dok3: 'keputusan' (bobot = 1.9163)
Dok4: 'mempercepat' (bobot = 1.9163)

ANALISIS SINGKAT:
Term dengan bobot TF-IDF tertinggi pada masing-masing dokumen cenderung merupakan kata
yang cukup sering muncul di dokumen tersebut namun jarang muncul di dokumen lain dalam
koleksi (df rendah), misalnya kata-kata topikal seperti jaringan, kecerdasan, atau
pola yang hanya muncul pada satu-dua dokumen. Sebaliknya, kata seperti sistem atau
komputer yang muncul di beberapa dokumen mendapat IDF lebih rendah sehingga bobot
TF-IDF-nya ditekan, meskipun frekuensi kemunculannya (TF) cukup tinggi. Hal ini
menunjukkan term-term tersebut penting karena bersifat diskriminatif: keberadaannya
membantu sistem membedakan topik satu dokumen dari dokumen lainnya, yang merupakan
fungsi utama dari pembobotan TF-IDF dalam sistem temu kembali informasi.

